# Project 3: Neural Net from Scratch

In this project, you will create a neural network by hand to perform binary
classification on a synthetic (made-up) dataset.  In this project, the data
is a simple 2-dimensional dataset so it can be easily plotted and visualized.

The goal of this project is not only to create the neural net from scratch, but
also to get practice training neural nets of various depths and seeing how
they work.


## Introduction

Let's examine our data set a little.

In [ ]:
# Read data and plot

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

alldata = np.loadtxt("output.csv", delimiter=",")
X = alldata[:, :2]
Y = alldata[:, 2:]

# Number of data points
m = 400

print("Red points:", np.sum(Y[:, 0] == 0))
print("Blue points:", np.sum(Y[:, 0] == 1))

# Plot X, colored by Y
plt.figure(figsize=(6, 6))
plt.scatter(X[Y[:, 0] == 0, 0], X[Y[:, 0] == 0, 1], color='red', label='Class 0')
plt.scatter(X[Y[:, 0] == 1, 0], X[Y[:, 0] == 1, 1], color='blue', label='Class 1')
plt.xlabel('x1')
plt.ylabel('x2')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
## Create testing and training sets.

# We have 400 data points total (200 per class).
# Use the first half for training and the second half for testing.

# So create a training set of rows 0-199,
# and a testing set of rows 200-399.
                      
X_train = None
X_test = None
Y_train = None
Y_test = None

# Sanity checks.
print(X_train.shape)  # should be (200, 2)
print(X_test.shape) # should be (200, 2)
print(Y_train.shape) # should be (200, 1)
print(Y_test.shape) # should be (200, 1)

In [ ]:
# Helper function to plot the decision boundary of a classifier.
# You will call this later, don't worry about it now.

def plot_decision_boundary(predict_fn, X, Y):
    """Plot the decision boundary of a classifier.
    
    predict_fn: a function that takes one row of X and returns 0 or 1.
    X: the full dataset features (used for axis limits and plotting points).
    Y: the full dataset labels.
    """
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 500),
        np.linspace(y_min, y_max, 500)
    )
    grid = np.c_[xx.ravel(), yy.ravel()]

    probs = np.array([predict_fn(row) for row in grid])
    if probs.ndim > 1 and probs.shape[1] == 1:
        probs = probs.ravel()
    Z = probs.reshape(xx.shape)

    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, levels=50, cmap="RdBu", alpha=0.6)
    plt.colorbar()
    plt.scatter(X[Y[:, 0] == 0, 0], X[Y[:, 0] == 0, 1], color='red', label='Class 0')
    plt.scatter(X[Y[:, 0] == 1, 0], X[Y[:, 0] == 1, 1], color='blue', label='Class 1')
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# Neural Network class.  The bulk of this project is writing a general-purpose
# python class that will allow a neural network of any number of layers and
# sizes of each layer.  You should fill in this class according to the backprop
# handout provided.

# The __init__ function and check_dimensions are already completed.
# The others require you to fill in some code.

import numpy as np
import scipy
sigmoid = scipy.special.expit
relu = lambda x: np.maximum(0, x)
relu_deriv = lambda x: (x > 0).astype(float)

class NeuralNet:
    
    def __init__(self, sizes: list[int]):
        """
        Initialize all of the variables we need to keep track of.
        W, b, Z, A, delta, deriv_W, and deriv_b are lists of matrices.
        """
        self.L = len(sizes)-1
        self.sizes = sizes
        self.W = [None] * (self.L+1)        # W[0] through W[L], but index 0 ignored
        self.b = [None] * (self.L+1)        # b[0] through b[L], but index 0 ignored
        self.Z = [None] * (self.L+1)        # Z[0] through Z[L], but index 0 ignored
        self.A = [None] * (self.L+1)        # A[0] through A[L], A[0] represents the input matrix X
        self.delta = [None] * (self.L+1)    # index 0 ignored
        self.deriv_W = [None] * (self.L+1)  # index 0 ignored
        self.deriv_b = [None] * (self.L+1)  # index 0 ignored

    def check_dimensions(self):
        """
        Print out dimensions of all the matrices - useful for debugging.
        """
        for ell in range(1, self.L+1):
            if self.W[ell] is not None: print(f"dim of W{ell} is", self.W[ell].shape)
            if self.b[ell] is not None: print(f"dim of b{ell} is", self.b[ell].shape) 
        for ell in range(1, self.L+1):
            if self.Z[ell] is not None: print(f"dim of Z{ell} is", self.Z[ell].shape) 
            if self.A[ell] is not None: print(f"dim of A{ell} is", self.A[ell].shape) 
        for ell in range(1, self.L+1):
            if self.delta[ell] is not None: print(f"dim of delta{ell} is", self.delta[ell].shape)
            if self.deriv_W[ell] is not None: print(f"dim of deriv_W{ell} is", self.deriv_W[ell].shape) 
            if self.deriv_b[ell] is not None: print(f"dim of deriv_b{ell} is", self.deriv_b[ell].shape) 

    def init_weights_randomly(self):
        """
        Set initial weights of W/b matrices randomly.
        """
        np.random.seed(0)  # for reproducibility
        for ell in range(1, self.L+1):
            W_rows = None
            W_cols = None
            b_length = None
            self.W[ell] = np.random.normal(0, 1, (W_rows, W_cols))
            self.b[ell] = np.random.normal(0, 1, (1, b_length))

    def forward_prop(self, X):
        """
        X is a matrix of features, m rows by n cols.
        returns: nothing
        
        This function should calculate and store all of the A and Z matrices appropriately.
        """
        pass

    def backward_prop(self, X, Y):
        """
        X is a matrix of features, m rows by n cols.
        Y is a matrix of correct outputs, m rows by n_L cols.
           Because this is binary classification, Y is (m by 1), and each output is 0 or 1.
        returns: nothing
           
        This function should calculate and store all of the delta, deriv_W, and deriv_b matrices appropriately.
        """
        pass

    def predict(self, x):
        """
        x is a vector of one input (one row from the X matrix).
        returns: probability of x being in the "1" class (float between 0 and 1).

        This function does the same thing as forward_prop, but for only one row of input,
        and returns the result as a single float.
        """
        pass

    def predict_01(self, x):
        """
        x is a vector of one input (one row from the X matrix).
        returns: 0 or 1

        This function does the same thing as predict, but rounds to 0 or 1.
        """
        pass

    def compute_cost(self, X, Y):
        """
        X is a matrix of features, m rows by n cols.
        Y is a matrix of correct outputs, m rows by n_L cols.
           Because this is binary classification, Y is (m by 1), and each output is 0 or 1.

        Return the total cost of putting all of the rows of X through the neural network, according
        to the cross-entropy loss function.  You can do this by calling self.predict on each row of X,
        or you can call self.forward_prop on X all at once and iterate over that.

        Hint: the cross-entropy formula takes log(y_hat), which will crash if y_hat is exactly
        0 or 1.  Use y_hat = np.clip(y_hat, 1e-7, 1 - 1e-7) to avoid this.
        """
        pass

In [ ]:
# Create a single-layer neural network, 2 features going to 1 output
nn1 = NeuralNet([2, 1])
nn1.init_weights_randomly()

In [ ]:
# Sanity checks for forward_prop and predict

nn1.check_dimensions()
nn1.init_weights_randomly()
nn1.forward_prop(X_train[0:3])  # put first 3 rows of X through the NN
print(nn1.Z[1])
print()
print(nn1.A[1])
print()
print(nn1.predict(X_train[0]))  # put first row through by itself, should match 


In [ ]:
# Sanity check for backprop.

nn1.check_dimensions()
nn1.init_weights_randomly()
nn1.backward_prop(X_train[0:3], Y_train[0:3])   # put first 3 rows of X through backprop

print(nn1.delta[1])
print()
print(nn1.deriv_W[1])
print()
print(nn1.deriv_b[1])
print()

In [ ]:
# Sanity check for compute cost

nn1.check_dimensions()
nn1.init_weights_randomly()
nn1.compute_cost(X_train, Y_train)

In [ ]:
# Write gradient descent:

# learning rate
ALPHA = None

# list of costs for each epoch as we compute them
J_sequence = []

nn1.init_weights_randomly()

# loop here

print("Final cost:", J_sequence[-1])
print("Final parameters:")
print(nn1.W)
print(nn1.b)

In [ ]:
# Let's plot the cost as a function of number of iterations of the
# gradient descent algorithm.

plt.scatter(range(0, len(J_sequence)), J_sequence)
plt.show()

In [ ]:
# Test/train accuracy - write this function

def compute_accuracy(X, Y):
    """
    Returns fraction of the examples in X that were classified correctly.
    You will want to call nn1.predict_01.
    """
    pass

# Testing and training accuracy for a 1-layer network should be close to 50%, but
# mathematically for this neural net can't get much higher.

print(compute_accuracy(X_train, Y_train))
print(compute_accuracy(X_test, Y_test))

In [ ]:
# Plot the decision boundary for nn1.
plot_decision_boundary(nn1.predict_01, X, Y)

# 2 layer neural net

Now we will repeat this for a 2-layer network.  You get to pick the number
of neurons in the hidden layer.

In [ ]:
# Create a 2-layer neural network, 
# 2 features going to (variable) hidden layer, going to 1 output
nn2 = NeuralNet([2, 4, 1])   # good place to start
nn2.init_weights_randomly()

In [ ]:
nn2.check_dimensions()
nn2.init_weights_randomly()
nn2.forward_prop(X_train[0:3])  # put first 3 rows of X through the NN
print(nn2.Z)
print()
print(nn2.A)
print()
print(nn2.predict(X_train[0]))  # put first row through by itself, should match 


In [ ]:
# Sanity check for backprop.

nn2.check_dimensions()
nn2.init_weights_randomly()
nn2.backward_prop(X_train[0:3], Y_train[0:3])   # put first 3 rows of X through backprop

print(nn2.delta)
print()
print(nn2.deriv_W)
print()
print(nn2.deriv_b)
print()

In [ ]:
# Sanity check for compute cost

nn2.check_dimensions()
nn2.init_weights_randomly()
nn2.compute_cost(X_train, Y_train)

In [ ]:
# Write gradient descent:

# learning rate
ALPHA = None

# list of costs
J_sequence = []

nn2.init_weights_randomly()

# write your gradient descent loop - don't forget to use "nn2" not "nn1"!

print("Final cost:", J_sequence[-1])
print("Final parameters:")
print(nn2.W)
print(nn2.b)

In [ ]:
# Let's plot the cost as a function of number of iterations of the
# gradient descent algorithm.

plt.scatter(range(0, len(J_sequence)), J_sequence)
plt.show()

In [ ]:
# Test/train accuracy - write this function

def compute_accuracy(X, Y):
    """
    Returns fraction of the examples in X that were classified correctly.
    You will want to call nn2.predict_01.
    """
    pass

# Testing and training accuracy for a 2-layer network can get easily over 90%.

print(compute_accuracy(X_train, Y_train))
print(compute_accuracy(X_test, Y_test))

In [ ]:
# Plot the decision boundary for nn2.
plot_decision_boundary(nn2.predict_01, X, Y)

In [ ]:
# Repeat the steps above with a 3-layer network, mostly just to show
# your code works with 3 layers.  Keep the number of neurons at each
# layer pretty small, try something like (2, 3, 3, 1).

# Copy all the code from the cells above to create your 3-layer network.
# Your final output must include gradient descent, the learning curve graph,
# the plot showing the "shape" of the predictions, and the compute_accuracy calculations.

# In the code that generates the 2-d decision boundary plot of the predictions, 
# you just need to call plot_decision_boundary(nn3.predict_01, X, Y).

# PyTorch Comparison

Now we will build the **exact same** neural network using PyTorch, copy in the same
initial random weights from your nn3, and train it with the same learning rate and
number of epochs. If your from-scratch code is correct, the loss curves and final
parameters should match.

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import torch.nn as nn

In [ ]:
# Save nn3's trained weights before the PyTorch section resets them.
nn3_trained_W = [w.copy() if w is not None else None for w in nn3.W]
nn3_trained_b = [b.copy() if b is not None else None for b in nn3.b]

In [ ]:
# Convert our NumPy data to PyTorch tensors.
# We use float64 to match NumPy's precision, so results are directly comparable.

X_train_t = torch.tensor(X_train, dtype=torch.float64)
X_test_t = torch.tensor(X_test, dtype=torch.float64)
y_train_t = torch.tensor(Y_train, dtype=torch.float64)
y_test_t = torch.tensor(Y_test, dtype=torch.float64)

print(X_train_t.shape, y_train_t.shape)
print(X_test_t.shape, y_test_t.shape)

In [ ]:
# Build a PyTorch model with the same architecture as nn3.
# nn.Sequential lets you stack layers in order.
# Use nn.Linear(in, out) for each layer, nn.ReLU() for hidden activations,
# and nn.Sigmoid() for the output.
# .double() converts the model to float64 to match NumPy's precision.

# **Follow the code given in pytorch-intro-classification in the binary classification section!**

model = nn.Sequential(
    # Fill in the layers to match nn3's architecture
    None
).double()  
# This double() is here to use 64-bit floats rather than 32-bit, to match NumPy's precision.
# Without it, your numbers from earlier in the notebook won't match PyTorch.

print(model)

In [ ]:
# Copy nn3's initial random weights into the PyTorch model.
#
# Important: PyTorch stores weights TRANSPOSED compared to our convention.
#   Our code:  W[ell] is (n_in, n_out) 
#   PyTorch:   weight is (n_out, n_in) 
#
# In nn.Sequential, the Linear layers are at indices 0, 2, 4
# (because ReLU/Sigmoid sit at indices 1, 3, 5).

# No code for you to fill in here, just run the cell.

nn3.init_weights_randomly()

with torch.no_grad():
    model[0].weight.copy_(torch.tensor(nn3.W[1].T))
    model[0].bias.copy_(torch.tensor(nn3.b[1].flatten()))
    model[2].weight.copy_(torch.tensor(nn3.W[2].T))
    model[2].bias.copy_(torch.tensor(nn3.b[2].flatten()))
    model[4].weight.copy_(torch.tensor(nn3.W[3].T))
    model[4].bias.copy_(torch.tensor(nn3.b[3].flatten()))

In [ ]:
# Train the PyTorch model with the same loss function, optimizer, learning rate,
# and number of epochs as your from-scratch nn3.

# **Follow the code given in pytorch-intro-classification in the binary classification section!**
# Hint: You will need BCELoss(), torch.optim.SGD, etc

# Save your costs/losses into a list called "losses" (this is what the pytorch-intro-classification
# notebook does).  We will use this losses list below.

# Write training loop here (same number of epochs as nn3)
# Remember to use X_train_t and y_train_t (tensor versions) not the numpy versions.

print("Final PyTorch loss:", losses[-1])
print()
print("Final parameters (transposed back to match our convention):")
print("W1:\n", model[0].weight.T.detach().numpy())
print("b1:\n", model[0].bias.detach().numpy())
print("W2:\n", model[2].weight.T.detach().numpy())
print("b2:\n", model[2].bias.detach().numpy())
print("W3:\n", model[4].weight.T.detach().numpy())
print("b3:\n", model[4].bias.detach().numpy())

In [ ]:
# Overlay the from-scratch and PyTorch loss curves — they should match!
plt.scatter(range(0, len(J_sequence)), J_sequence, label='Manual gradient descent')
plt.plot(range(len(losses)), losses, label='PyTorch', alpha=0.7, linestyle='-', color='red')
plt.xlabel('Epoch')
plt.ylabel('Cost')
plt.legend()
plt.title('Loss curves should overlap')
plt.show()

In [ ]:
# Compare final weights side by side: from-scratch (nn3) vs PyTorch

print("--- Layer 1 ---")
print("W1 (from scratch):\n", nn3_trained_W[1])
print("W1 (PyTorch):\n", model[0].weight.T.detach().numpy())
print()
print("b1 (from scratch):\n", nn3_trained_b[1])
print("b1 (PyTorch):\n", model[0].bias.detach().numpy())
print()

print("--- Layer 2 ---")
print("W2 (from scratch):\n", nn3_trained_W[2])
print("W2 (PyTorch):\n", model[2].weight.T.detach().numpy())
print()
print("b2 (from scratch):\n", nn3_trained_b[2])
print("b2 (PyTorch):\n", model[2].bias.detach().numpy())
print()

print("--- Layer 3 ---")
print("W3 (from scratch):\n", nn3_trained_W[3])
print("W3 (PyTorch):\n", model[4].weight.T.detach().numpy())
print()
print("b3 (from scratch):\n", nn3_trained_b[3])
print("b3 (PyTorch):\n", model[4].bias.detach().numpy())

In [ ]:
# Accuracy from the PyTorch model

model.eval()

def compute_accuracy_torch(X_data, y_data):
    with torch.no_grad():
        y_hat = model(X_data)
        predictions = (y_hat >= 0.5).float()
        accuracy = (predictions == y_data).float().mean()
    return accuracy.item()

print("Train accuracy:", compute_accuracy_torch(X_train_t, y_train_t))
print("Test accuracy: ", compute_accuracy_torch(X_test_t, y_test_t))

In [ ]:
# Decision boundary from the PyTorch model

def torch_predict_01(x):
    """Wrapper so we can pass the PyTorch model to plot_decision_boundary."""
    with torch.no_grad():
        t = torch.tensor(x, dtype=torch.float64).unsqueeze(0)
        return (model(t) >= 0.5).int().item()

plot_decision_boundary(torch_predict_01, X, Y)

<big>Congratulations!  You have written a complete neural network from end to end, trained
it with backpropagation, and verified it works the **exact** same way that PyTorch does!</big>